# 43 — Supervisor-Worker

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/43-supervisor-worker/supervisor_worker_workbook.ipynb)

## What you'll learn

- **`with_structured_output(WorkerRoute)`** — how a supervisor LLM emits a Pydantic model instead of free text, making routing decisions machine-readable and validated.
- **Pydantic-enforced worker selection** — `Literal['researcher','summarizer','analyst']` guarantees the LLM can only name valid workers; invalid names are rejected at parse time.
- **Dynamic worker dispatch** — `make_worker(worker_type)` returns a closure that injects the correct system prompt, so one factory function creates all three workers.
- **Supervisor rationale field** — the `rationale` field in `WorkerRoute` makes routing *explainable*: you can log or display why the supervisor chose a particular worker.

### How this compares to related examples

| Example | Pattern | Key difference |
|---------|---------|----------------|
| **6-multi-agent-graph** | `add_messages` reducer, agent nodes | Message-passing architecture; agents communicate via shared message list |
| **38-langgraph-command-pattern** | `Command(goto=route)` | Routing expressed as an imperative command returned from the node itself |
| **This example (43)** | `with_structured_output(WorkerRoute)` | Routing expressed as a validated Pydantic schema; rationale is explicit |

In [ ]:
# Install dependencies (Colab guard)
import sys

if "google.colab" in sys.modules:
    %pip install -q langchain langchain-openai langgraph python-dotenv pydantic

In [ ]:
# API key setup
import os

if "google.colab" in sys.modules:
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
else:
    from dotenv import load_dotenv

    load_dotenv()

assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY before running"

In [ ]:
# Concept: why structured output beats string parsing for routing
#
# String parsing approach (fragile):
#   response = llm.invoke(prompt)
#   if 'researcher' in response.content.lower(): route = 'researcher'
#   elif 'summarizer' in ...: ...
#   else: route = 'researcher'  # silent fallback hides errors
#
# Structured output approach (robust):
#   class WorkerRoute(BaseModel):
#       worker: Literal['researcher', 'summarizer', 'analyst']  # enforced by Pydantic
#       rationale: str                                           # explainable routing
#   route = router_llm.invoke(prompt)   # raises ValidationError if LLM hallucinates
#   route.worker  # guaranteed to be one of the three valid values
#
# Benefits:
#   1. Literal type enforcement - Pydantic rejects invalid worker names at parse time
#   2. Rationale field - log WHY the supervisor chose a worker, not just which one
#   3. No silent fallbacks - errors surface immediately, not in production
#   4. Schema is self-documenting - the type hint IS the routing contract

print('Routing approach: structured output -> Literal["researcher","summarizer","analyst"]')

In [ ]:
# Full self-contained implementation
from typing import Literal, TypedDict

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, StateGraph
from pydantic import BaseModel

# --- Schema ---


class WorkerRoute(BaseModel):
    worker: Literal["researcher", "summarizer", "analyst"]
    rationale: str


class SupervisorState(TypedDict):
    task: str
    worker_type: str
    rationale: str
    worker_result: str


# --- Prompts ---

WORKER_PROMPTS = {
    "researcher": "You are a researcher. Provide 3-5 factual, well-sourced points about: {task}",
    "summarizer": "You are a summarizer. Give a concise 3-bullet summary of key points about: {task}",
    "analyst": "You are an analyst. Provide a structured analysis with pros/cons/recommendation for: {task}",
}

SUPERVISOR_PROMPT = (
    "You are a supervisor. Classify this task to the best worker: "
    "researcher (needs facts/research), summarizer (needs condensed info), "
    "analyst (needs structured analysis). Task: {task}"
)

# --- LLMs ---

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)
router_llm = llm.with_structured_output(WorkerRoute)

# --- Nodes ---


def supervisor(state: SupervisorState) -> dict:
    result = router_llm.invoke([HumanMessage(content=SUPERVISOR_PROMPT.format(task=state["task"]))])
    print(f"  Supervisor -> {result.worker}")
    print(f"  Rationale: {result.rationale[:80]}")
    return {"worker_type": result.worker, "rationale": result.rationale}


def make_worker(worker_type: str):
    def worker(state: SupervisorState) -> dict:
        prompt = WORKER_PROMPTS[worker_type].format(task=state["task"])
        result = llm.invoke([SystemMessage(content=prompt)])
        return {"worker_result": result.content}

    worker.__name__ = f"{worker_type}_worker"
    return worker


def route(state: SupervisorState) -> str:
    return state["worker_type"]


# --- Graph ---


def create_workflow():
    graph = StateGraph(SupervisorState)
    graph.add_node("supervisor", supervisor)
    for wt in WORKER_PROMPTS:
        graph.add_node(wt, make_worker(wt))
        graph.add_edge(wt, END)
    graph.set_entry_point("supervisor")
    graph.add_conditional_edges("supervisor", route, {wt: wt for wt in WORKER_PROMPTS})
    return graph.compile()


app = create_workflow()
print("Graph compiled successfully")

In [ ]:
# Run sample tasks
SAMPLE_TASKS = [
    "Research the latest developments in quantum computing.",
    "Summarize the key points of the React documentation.",
    "Analyze the pros and cons of microservices vs monoliths.",
]

for task in SAMPLE_TASKS:
    print(f"\nTASK: {task}")
    print("-" * 60)
    result = app.invoke({"task": task, "worker_type": "", "rationale": "", "worker_result": ""})
    print(f"Worker: {result['worker_type']}")
    print(f"Result preview:\n{result['worker_result'][:300]}")

## Exercises

1. **Add a 4th worker `coder`** — extend `WORKER_PROMPTS` with a prompt like `"Write a short Python script that demonstrates: {task}"`, add `'coder'` to the `Literal` in `WorkerRoute`, and add the node/edge in `create_workflow`.

2. **Add a feedback loop** — after a worker completes, route back to the supervisor with a grading prompt. If the supervisor scores the result below a threshold (e.g. < 7/10), re-route to a different worker. Add a `grade` field to `SupervisorState` and a conditional edge from each worker back to `supervisor`.

3. **Parallel workers** — use the LangGraph `Send` API to route to *two* workers simultaneously, then add an `aggregator` node that merges both results. Compare output quality vs. single-worker routing.

4. **Worker confidence scoring** — add a `confidence: float` field to `WorkerRoute`. If `confidence < 0.6`, log a warning and optionally re-route to a fallback worker.

---

## What's next

- **[6-multi-agent-graph](../6-multi-agent-graph/)** — message-passing multi-agent architecture with `add_messages` reducer
- **[19-multi-agent-notebook](../19-multi-agent-notebook/)** — multi-agent coordination patterns in a notebook format
- **[38-langgraph-command-pattern](../38-langgraph-command-pattern/)** — `Command(goto=route)` as an alternative routing primitive